In [1]:
#%%
import pandas as pd
import numpy as np

# ------------------------- STEP 1: LOAD DATA -------------------------
#%%
# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")
# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")
# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")
# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")
# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")
# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')

#%%
# Define mapping for admission types
emergency_types = ['DIRECT EMER.', 'EW EMER.', 'URGENT']
normal_types = ['AMBULATORY OBSERVATION', 'DIRECT OBSERVATION', 'ELECTIVE', 
                'EU OBSERVATION', 'OBSERVATION ADMIT', 'SURGICAL SAME DAY ADMISSION']

# Create a new column 'admission_category' based on the mapping
admissions['admission_category'] = admissions['admission_type'].apply(
    lambda x: 'EMERGENCY' if x in emergency_types else 'NORMAL'
)

# Check if mapping worked correctly
print(admissions['admission_category'].value_counts())

/tmp/ipykernel_5438/1455114467.py:18: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")


admission_category
NORMAL       291667
EMERGENCY    254361
Name: count, dtype: int64


In [2]:
# Read codes
ccs_dx_mapping = pd.read_csv("/root/MIMICIV/src/codes/CCS_DX_mapping.csv")
ccs_dx_categories = pd.read_csv("/root/MIMICIV/src/codes/CCS_DX_categories.csv")
ccs_pcs_categories = pd.read_csv("/root/MIMICIV/src/codes/CCS_PCS_categories.csv")
ccs_pcs_mapping = pd.read_csv("/root/MIMICIV/src/codes/CCS_PCS_mapping.csv")

In [3]:
len(ccs_dx_mapping['category_code'].unique())

308

In [4]:
len(ccs_dx_categories)

308

In [5]:
ccs_dx = pd.merge(ccs_dx_mapping, ccs_dx_categories, how='inner', on='category_code')
ccs_dx.head()

,category_code,code,vocabulary_id,code_chapter,L2code,category_desc
0,4,1120,ICD9CM,Infectious and parasitic diseases,1.2,Mycoses
1,4,1100,ICD9CM,Infectious and parasitic diseases,1.2,Mycoses
2,4,1101,ICD9CM,Infectious and parasitic diseases,1.2,Mycoses
3,4,1102,ICD9CM,Infectious and parasitic diseases,1.2,Mycoses
4,4,1103,ICD9CM,Infectious and parasitic diseases,1.2,Mycoses


In [10]:
ccs_dx['vocabulary_id'].unique()
ccs_dx['vocabulary_id'] = ccs_dx['vocabulary_id'].replace(
    {
    'ICD9CM': '9',
    'ICD10CM': '10'
    })
ccs_dx['key'] = ccs_dx['code'].astype(str) + '_' + ccs_dx['vocabulary_id']

In [11]:
ccs_dx.head()

,category_code,code,vocabulary_id,code_chapter,L2code,category_desc,key
0,4,1120,9,Infectious and parasitic diseases,1.2,Mycoses,1120_9
1,4,1100,9,Infectious and parasitic diseases,1.2,Mycoses,1100_9
2,4,1101,9,Infectious and parasitic diseases,1.2,Mycoses,1101_9
3,4,1102,9,Infectious and parasitic diseases,1.2,Mycoses,1102_9
4,4,1103,9,Infectious and parasitic diseases,1.2,Mycoses,1103_9


In [12]:
ccs_dx_clean = ccs_dx.drop_duplicates(subset='key', keep='first')

In [13]:
# ------------------------- STEP 2: MAP ICD CODES TO DESCRIPTIONS -------------------------

# Merge diagnoses with ICD descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on=["icd_code", "icd_version"], how="left")
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with ICD descriptions
procedures = procedures.merge(icd_procedures, on=["icd_code", "icd_version"], how="left")
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

# Keep only necessary columns
diagnoses = diagnoses[['subject_id', 'hadm_id', 'diagnosis_description', 'icd_code', 'icd_version']]
procedures = procedures[['subject_id', 'hadm_id', 'procedure_description', 'icd_code', 'icd_version']]


In [14]:
diagnoses.head()

,subject_id,hadm_id,diagnosis_description,icd_code,icd_version
0,10000032,22595853,Portal hypertension,5723,9
1,10000032,22595853,Other ascites,78959,9
2,10000032,22595853,Cirrhosis of liver without mention of alcohol,5715,9
3,10000032,22595853,Unspecified viral hepatitis C without hepatic ...,07070,9
4,10000032,22595853,"Chronic airway obstruction, not elsewhere clas...",496,9


In [15]:
len(diagnoses)

6364488

In [16]:
diagnoses['key'] = diagnoses['icd_code'].astype(str) + '_' + diagnoses['icd_version'].astype(str)

In [17]:
diagnoses.head()

,subject_id,hadm_id,diagnosis_description,icd_code,icd_version,key
0,10000032,22595853,Portal hypertension,5723,9,5723_9
1,10000032,22595853,Other ascites,78959,9,78959_9
2,10000032,22595853,Cirrhosis of liver without mention of alcohol,5715,9,5715_9
3,10000032,22595853,Unspecified viral hepatitis C without hepatic ...,07070,9,07070_9
4,10000032,22595853,"Chronic airway obstruction, not elsewhere clas...",496,9,496_9


In [18]:
len(diagnoses)

6364488

In [19]:
diagnoses_ccs = diagnoses.merge(ccs_dx, how='left', on='key')

In [20]:
diagnoses_ccs_clean = diagnoses.merge(ccs_dx_clean, how='left', on='key')

In [21]:
diagnoses_ccs.head()

,subject_id,hadm_id,diagnosis_description,icd_code,icd_version,key,category_code,code,vocabulary_id,code_chapter,L2code,category_desc
0,10000032,22595853,Portal hypertension,5723,9,5723_9,151.0,5723,9,Diseases of the digestive system,9.8,Other liver diseases
1,10000032,22595853,Other ascites,78959,9,78959_9,151.0,78959,9,Diseases of the digestive system,9.8,Other liver diseases
2,10000032,22595853,Cirrhosis of liver without mention of alcohol,5715,9,5715_9,151.0,5715,9,Diseases of the digestive system,9.8,Other liver diseases
3,10000032,22595853,Unspecified viral hepatitis C without hepatic ...,07070,9,07070_9,6.0,07070,9,Infectious and parasitic diseases,1.3,Hepatitis
4,10000032,22595853,"Chronic airway obstruction, not elsewhere clas...",496,9,496_9,127.0,496,9,Diseases of the respiratory system,8.2,Chronic obstructive pulmonary disease and bron...


In [22]:
diagnoses_ccs[pd.isna(diagnoses_ccs['vocabulary_id'])]

,subject_id,hadm_id,diagnosis_description,icd_code,icd_version,key,category_code,code,vocabulary_id,code_chapter,L2code,category_desc
39,10000032,29079034,Do not resuscitate status,V4986,9,V4986_9,NaN,NaN,NaN,NaN,NaN,NaN
83,10000161,22148160,"Headache, unspecified",R519,10,R519_10,NaN,NaN,NaN,NaN,NaN,NaN
86,10000248,20600184,Fall from snowboard,E8854,9,E8854_9,NaN,NaN,NaN,NaN,NaN,NaN
87,10000248,20600184,Street and highway accidents,E8495,9,E8495_9,NaN,NaN,NaN,NaN,NaN,NaN
103,10000635,20642640,Contact with and (suspected) exposure to COVID-19,Z20822,10,Z20822_10,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
6491380,19999784,24240036,Elevation of levels of liver transaminase levels,R7401,10,R7401_10,NaN,NaN,NaN,NaN,NaN,NaN
6491410,19999784,25127296,Contact with and (suspected) exposure to COVID-19,Z20822,10,Z20822_10,NaN,NaN,NaN,NaN,NaN,NaN
6491432,19999784,25989171,Elevation of levels of liver transaminase levels,R7401,10,R7401_10,NaN,NaN,NaN,NaN,NaN,NaN
6491458,19999784,27302283,Elevation of levels of liver transaminase levels,R7401,10,R7401_10,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
diagnoses_ccs[pd.isna(diagnoses_ccs['icd_version'])]

,subject_id,hadm_id,diagnosis_description,icd_code,icd_version,key,category_code,code,vocabulary_id,code_chapter,L2code,category_desc


In [140]:
len(diagnoses_ccs)- len(diagnoses)

127118

In [24]:
len(diagnoses_ccs_clean) - len(diagnoses)

0

In [25]:
ccs_dx[ccs_dx['key'].duplicated(keep=False)]

,category_code,code,vocabulary_id,code_chapter,L2code,category_desc,key
3313,652,31200,9,Mental Illness,5.30,Attention-deficit conduct and disruptive behav...,31200_9
3314,652,31201,9,Mental Illness,5.30,Attention-deficit conduct and disruptive behav...,31201_9
3315,652,31202,9,Mental Illness,5.30,Attention-deficit conduct and disruptive behav...,31202_9
3316,652,31203,9,Mental Illness,5.30,Attention-deficit conduct and disruptive behav...,31203_9
3317,652,31210,9,Mental Illness,5.30,Attention-deficit conduct and disruptive behav...,31210_9
...,...,...,...,...,...,...,...
37824,6709,V403,9,Mental Illness,5.15,Other miscellaneous mental conditions,V403_9
37825,6709,V4031,9,Mental Illness,5.15,Other miscellaneous mental conditions,V4031_9
37826,6709,V4039,9,Mental Illness,5.15,Other miscellaneous mental conditions,V4039_9
37827,6709,V409,9,Mental Illness,5.15,Other miscellaneous mental conditions,V409_9


In [26]:
ccs_dx[ccs_dx['key'] == '31200_9']

,category_code,code,vocabulary_id,code_chapter,L2code,category_desc,key
3313,652,31200,9,Mental Illness,5.3,Attention-deficit conduct and disruptive behav...,31200_9
37586,6521,31200,9,Mental Illness,5.3,Conduct disorder,31200_9


In [27]:
ccs_dx_mapping[ccs_dx_mapping['code'] == '31200']

,category_code,code,vocabulary_id
1041,652,31200,ICD9CM
8344,6521,31200,ICD9CM


In [28]:
ccs_dx_categories[ccs_dx_categories['category_code'] == 6521]

,code_chapter,category_code,L2code,category_desc
283,Mental Illness,6521,5.3,Conduct disorder


In [29]:
diagnoses_ccs_clean.head()

,subject_id,hadm_id,diagnosis_description,icd_code,icd_version,key,category_code,code,vocabulary_id,code_chapter,L2code,category_desc
0,10000032,22595853,Portal hypertension,5723,9,5723_9,151.0,5723,9,Diseases of the digestive system,9.8,Other liver diseases
1,10000032,22595853,Other ascites,78959,9,78959_9,151.0,78959,9,Diseases of the digestive system,9.8,Other liver diseases
2,10000032,22595853,Cirrhosis of liver without mention of alcohol,5715,9,5715_9,151.0,5715,9,Diseases of the digestive system,9.8,Other liver diseases
3,10000032,22595853,Unspecified viral hepatitis C without hepatic ...,07070,9,07070_9,6.0,07070,9,Infectious and parasitic diseases,1.3,Hepatitis
4,10000032,22595853,"Chronic airway obstruction, not elsewhere clas...",496,9,496_9,127.0,496,9,Diseases of the respiratory system,8.2,Chronic obstructive pulmonary disease and bron...


In [39]:
# If missing 'code', fill it with 'icd_code'
# This is to ensure that we have a code for each diagnosis, even if it doesn't map
diagnoses_ccs_clean['code'] = diagnoses_ccs_clean['code'].fillna(diagnoses_ccs_clean['icd_code'])
diagnoses_ccs_clean['category_code'] = diagnoses_ccs_clean['category_code'].fillna(diagnoses_ccs_clean['icd_code'])


In [31]:
# If missing 'category_desc', fill it with 'diagnosis_description'
# This is to ensure that we have a description for each diagnosis, even if it doesn't map
diagnoses_ccs_clean['category_desc'] = diagnoses_ccs_clean['category_desc'].fillna(diagnoses_ccs_clean['diagnosis_description'])

In [ ]:
no_map_diagnoses = diagnoses_ccs_clean[pd.isna(diagnoses_ccs_clean['code_chapter'])][['icd_code', 'icd_version','category_desc']].drop_duplicates()
#pd.DataFrame(no_map_diagnoses, columns=['icd_code', 'icd_version','category_desc']).to_csv("/root/MIMICIV/src/codes/no_map_diagnoses.csv", index=False)

In [57]:
no_map_diagnoses

,icd_code,icd_version,category_desc
34,V4986,9,Do not resuscitate status
77,R519,10,"Headache, unspecified"
80,E8854,9,Fall from snowboard
81,E8495,9,Street and highway accidents
96,Z20822,10,Contact with and (suspected) exposure to COVID-19
...,...,...,...
6233198,L8996,10,Pressure-induced deep tissue damage of unspeci...
6251817,V1587,9,History of extracorporeal membrane oxygenation...
6257397,32712,9,Idiopathic hypersomnia without long sleep time
6290148,32719,9,Other organic hypersomnia


In [51]:
diagnoses_ccs_clean[pd.isna(diagnoses_ccs_clean['code_chapter'])].head()

,subject_id,hadm_id,diagnosis_description,icd_code,icd_version,key,category_code,code,vocabulary_id,code_chapter,L2code,category_desc
34,10000032,29079034,Do not resuscitate status,V4986,9,V4986_9,V4986,V4986,NaN,NaN,NaN,Do not resuscitate status
77,10000161,22148160,"Headache, unspecified",R519,10,R519_10,R519,R519,NaN,NaN,NaN,"Headache, unspecified"
80,10000248,20600184,Fall from snowboard,E8854,9,E8854_9,E8854,E8854,NaN,NaN,NaN,Fall from snowboard
81,10000248,20600184,Street and highway accidents,E8495,9,E8495_9,E8495,E8495,NaN,NaN,NaN,Street and highway accidents
96,10000635,20642640,Contact with and (suspected) exposure to COVID-19,Z20822,10,Z20822_10,Z20822,Z20822,NaN,NaN,NaN,Contact with and (suspected) exposure to COVID-19


In [43]:
len(diagnoses_ccs_clean['category_code'].unique())

1506

In [44]:
len(diagnoses_ccs_clean['icd_code'].unique())

28562

In [ ]:
# need to do the same for procedures
